# NVIDIA Ising color-code pre-decoder tutorial

This tutorial covers the **color-code** member of the [NVIDIA Ising
pre-decoder](https://github.com/NVIDIA/Ising-Decoding) family. It is the
counterpart to the surface-code walkthrough in
[`predecoder.ipynb`](./predecoder.ipynb): the model, training pipeline, and
deployment story are the same 3D fully-convolutional pre-decoder, but the code
is a **triangular color code** and the downstream global decoder is
[Chromobius](https://github.com/quantumlib/chromobius) instead of PyMatching.

If you are new to the pre-decoder idea, read the *What is a pre-decoder?*
section of the surface-code tutorial first — this notebook assumes that
background and focuses on what is **different for color codes**.

## What is a color-code pre-decoder?

A color-code memory experiment produces detector syndromes across space **and**
time, exactly as the surface code does. The pre-decoder is a 3D convolutional
neural network that consumes those syndromes and predicts local space-time
corrections that **sparsify the input syndromes**, plus a partial logical
correction. A standard color-code
decoder (**Chromobius**) then makes the final logical decision on the much
sparser residual:

```text
  color-code syndromes ──▶ Ising pre-decoder (CNN) ──▶ residual syndrome ──▶ Chromobius ──▶ logical correction
                                    │                                                              ▲
                                    └────────────── partial logical correction ──────────────────┘ (XOR)
```

Because the network removes most detector events before Chromobius runs, the
global decode is faster and — at larger distances and lower physical error
rates — more accurate.

### What you will learn

1. **Quick Start** — Run a trained **Color-Code Model 5** pre-decoder
   end-to-end and compare pre-decoder + Chromobius against Chromobius alone.
2. **Training** — Train your own color-code pre-decoder, including the
   augmented-DEM precompute step and the color-specific defaults.
3. **Evaluation & inference optimization** — LER / SDR / decode-time sweeps and
   how the same architecture maps onto the ONNX/TensorRT deployment path.

The canonical reference for every command here is the **Color code support**
section of the repository [README](../README.md#color-code-support); this
notebook is a guided tour of that path.


### Setup

This tutorial lives in the `cookbook/` directory of the `Ising-Decoding`
repository.

**Prerequisites:**
- **NVIDIA GPU** with CUDA drivers installed (`nvidia-smi` must be on your PATH)
- **Python 3.11, 3.12, or 3.13**

The cells below will:
1. Locate the repository root and add the predecoder source code to the Python path
2. Detect the CUDA version from your GPU driver and install the matching PyTorch build
3. Install the predecoder dependencies. Color-code **inference** needs `stim` and
   [`chromobius`](https://github.com/quantumlib/chromobius) (both in
   `requirements_public_inference.txt`); color-code **training** additionally
   needs cuQuantum/cuStabilizer for on-the-fly data generation (the training
   requirements file is a superset and is what we install here).

**Note:** Color-code inference and training run on the GPU path. There is no CPU
fast-path; expect the Quick Start cell to require a CUDA device.


In [1]:
import subprocess, sys, os, re, shutil

NOTEBOOK_DIR = os.path.abspath('')
PREDECODER_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR,'..'))
sys.path.insert(0, os.path.join(PREDECODER_ROOT, 'code'))

print(f'PREDECODER_ROOT: {PREDECODER_ROOT}')
assert os.path.isdir(os.path.join(PREDECODER_ROOT, 'code')), (
    f"Cannot find predecoder source code at {PREDECODER_ROOT}/code. "
    f"This notebook must live at <repo_root>/cookbook/."
)

assert shutil.which('nvidia-smi'), 'nvidia-smi not found — this tutorial requires an NVIDIA GPU.'

nvsmi_output = subprocess.check_output(['nvidia-smi'], text=True)
cuda_match = re.search(r'CUDA Version:\s+([\d.]+)', nvsmi_output)
assert cuda_match, 'Could not detect CUDA version from nvidia-smi output.'
cuda_ver = cuda_match.group(1)
cuda_major = cuda_ver.split('.')[0]
print(f'CUDA {cuda_ver} detected (major: {cuda_major})')

def _pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

print('Environment OK.')


PREDECODER_ROOT: /workspace
CUDA 12.8 detected (major: 12)
Environment OK.


In [ ]:
_pip('--upgrade', 'pip', 'setuptools<82', 'wheel')

torch_cuda_tag = {'12': 'cu128', '13': 'cu130'}[cuda_major]
print(f'Installing PyTorch (wheel index: {torch_cuda_tag})...')
_pip('torch', '--index-url', f'https://download.pytorch.org/whl/{torch_cuda_tag}',
     '--extra-index-url', 'https://pypi.org/simple')

import torch
assert torch.cuda.is_available(), 'PyTorch installed but CUDA is not available.'
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}, '
      f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
train_req = os.path.join(PREDECODER_ROOT, 'code', f'requirements_public_train-cu{cuda_major}.txt')
assert os.path.exists(train_req), (
    f"No training requirements for CUDA {cuda_major}: {train_req}\n"
    f"Available: requirements_public_train-cu12.txt, requirements_public_train-cu13.txt"
)
print(f'Installing predecoder dependencies from: {os.path.basename(train_req)}')
print('  (includes: stim, pymatching, chromobius, cuquantum, and more)')
_pip('-r', train_req)

# Color-code inference uses Chromobius as the global decoder; make sure it imported.
import chromobius, stim
print(f'chromobius {getattr(chromobius, "__version__", "?")}, stim {stim.__version__} ready.')
print('All dependencies installed.')


All color-code work flows through the same public runner as the surface
code (`code/workflows/run.py`). For an interactive tutorial we call the runner's
building blocks directly:

- `OmegaConf.load(...)` reads the shipped inference config
  (`conf/config_inference_color_model_5.yaml`).
- `DistributedManager` sets up the (single-GPU here) device context.
- `_load_model(cfg, dist)` builds the architecture and loads the checkpoint
  at `cfg.model_checkpoint_file` (point it at your trained `.pt`).
- `count_logical_errors_color(...)` samples color-code syndromes, runs the
  pre-decoder, decodes the residual with Chromobius, and returns the logical
  error rate for the pre-decoder pipeline **and** the Chromobius-only baseline.


In [2]:
import numpy as np
import torch
from omegaconf import OmegaConf

from training.distributed import DistributedManager
from workflows.run import _load_model
from evaluation.logical_error_rate_color import count_logical_errors_color
from model.registry import get_model_spec
from model.factory import ModelFactory


---
## Quick Start

The repository ships a standalone inference config
(`conf/config_inference_color_model_5.yaml`) pinned to the **Color-Code
Model 5** architecture. Train such a checkpoint with the configs in the
Training section below (or bring your own Model-5-shaped `.pt`) and point the
config's `model_checkpoint_file` at it. The cell outputs kept in this notebook
are from a run with a trained Model 5 checkpoint.

The Quick Start below:

- **Syndrome data generation** — a noisy triangular color-code memory circuit is
  simulated with Stim (superdense, nearest-neighbor schedule), producing
  detector bits and a logical observable per shot.
- **Trained model** — the 6-layer 3D CNN
  (`[256, 256, 256, 256, 256, 4]`, ~7.1 M parameters) reduces syndrome density.
- **Chromobius on residuals** — Chromobius decodes the sparser residual
  syndrome; the final logical prediction is XOR'd with the pre-decoder's partial
  correction.

`count_logical_errors_color` runs all of this and reports the post-pre-decoder
logical error rate alongside the Chromobius-only baseline on identical shots.

We start from the shipped inference config and override only the evaluation
window for a quick run (`distance = n_rounds = 13`, a single basis, a modest shot
count). The model is fully convolutional, so the same checkpoint also evaluates
at any other distance.


In [3]:
# ── Load the shipped color-code inference config ──────────────────────────────
cfg = OmegaConf.load(os.path.join(PREDECODER_ROOT, "conf", "config_inference_color_model_5.yaml"))

# The config's default checkpoint path is repo-relative; resolve it from the
# repo root — or point it directly at your trained checkpoint:
#   cfg.model_checkpoint_file = "/path/to/your/checkpoint.pt"
cfg.model_checkpoint_file = os.path.join(PREDECODER_ROOT, cfg.model_checkpoint_file)

# Quick-run overrides: evaluation window + shot count.
# d = n_rounds = 13 matches the trained receptive field and shows a clear win in
# a few seconds. The model is fully convolutional, so the same checkpoint also
# evaluates at d = 5..31; larger distances and lower p give the biggest gains
# but take longer to sample.
cfg.test.distance        = 13
cfg.test.n_rounds        = 13
cfg.test.p_error         = 0.003
cfg.test.meas_basis_test = "X"      # "X", "Z", or "both"
cfg.test.num_samples     = 8192     # raise for tighter error bars

# In a container the default DataLoader workers can exhaust /dev/shm; run the
# eval loader in-process. (Equivalent to PREDECODER_INFERENCE_NUM_WORKERS=0; see
# the inference note in the README.) On bare metal you can keep the defaults.
cfg.test.dataloader.num_workers        = 0
cfg.test.dataloader.persistent_workers = False
cfg.test.dataloader.prefetch_factor    = None

# ── Device context + model load ───────────────────────────────────────────────
DistributedManager.initialize()
dist  = DistributedManager()
model = _load_model(cfg, dist)      # builds the CNN and loads the checkpoint

# ── Sample syndromes, run pre-decoder + Chromobius, compute LER ───────────────
result = count_logical_errors_color(
    model, dist.device, dist, cfg,
    include_diagnostics=False,   # set True for the full Chromobius timing breakdown
    log_summary=True,
)

print("\nColor-Code Model 5 — inference result")
for basis, data in result.items():
    print(f"\n{basis}-basis:")
    for key, value in data.items():
        print(f"  {key}: {value}")

    # The pre-decoder LER and the Chromobius-only baseline are reported under
    # keys that contain "logical_error_rate" and "chromobius_error_rate"; pull
    # the mean of each (key names are matched loosely so this survives minor
    # schema changes) and print the improvement factor.
    def _mean(d, needle):
        for k, v in d.items():
            if needle in k and "mean" in k and isinstance(v, (int, float)):
                return v
        return None
    pd_ler = _mean(data, "logical_error_rate")
    cb_ler = _mean(data, "chromobius_error_rate")
    if pd_ler is not None and cb_ler is not None and pd_ler > 0:
        print(f"  -> improvement (Chromobius / pre-decoder): {cb_ler / pd_ler:.1f}x")


Loading model for task: inference
Loading model from: /workspace/models/Ising-Decoder-ColorCode-5.pt
Model loaded (7,134,468 parameters)
[Color Code LER] Basis: X, p_error: 0.003
[Color Code LER] noise_model_family: legacy, noise_instruction_semantics: current, test.noise_model: none, gidney_style_noise: False
[Color Code LER] Using physical-frame observable (data-only parity)
[Color Code LER] delta_s2 conversion: disabled (default)
[Color Code LER] Time taken: 4.896s  |  PD+Chromobius=4.9767e-04 ± 6.8e-05 (53/8192)  |  Chromobius=3.3710e-03 ± 1.7e-04 (359/8192)

Color-Code Model 5 — inference result

X-basis:
  num_shots: 8192
  n_rounds: 13
  logical_errors: 53
  chromobius_errors: 359
  logical_error_rate (mean): 0.0004976712740384615
  logical_error_rate (stderr): 6.813891145792395e-05
  chromobius_error_rate (mean): 0.0033710186298076925
  chromobius_error_rate (stderr): 0.00017397346759678725
  -> improvement (Chromobius / pre-decoder): 6.8x


At `d = 13, p = 0.003` the pre-decoder already lowers the logical error rate
per round **several-fold** over Chromobius alone (≈4–7× here; the exact factor
fluctuates run-to-run at these low error counts — raise `num_samples` to tighten
it), while removing ~98% of the detector events before Chromobius runs (a
corresponding decode-time speedup). The advantage grows with code distance and
with lower physical error rates, where the syndrome is sparse and most detector
events are locally correctable.

The [README results section](../README.md#results) reports the full sweep
(`n_rounds = d`, X and Z bases, `d = 5..31`). A few X-basis points:

| Distance | p | Chromobius | PD + Chromobius | Improvement |
|---|---|---|---|---|
| 13 | 0.002 | 3.70e-04 | 6.29e-05 | 5.9× |
| 21 | 0.002 | 3.60e-05 | 9.94e-07 | 36× |
| 31 | 0.002 | 1.83e-06 | 5.13e-09 | 356× |

To reproduce points like these, raise `cfg.test.distance` / `cfg.test.n_rounds`
and `cfg.test.num_samples`, and lower `cfg.test.p_error`.


---
## Model & architecture

The Quick Start's inference config targets **model 5** in the registry: a
6-layer 3D fully-convolutional residual network with channel widths
`[256, 256, 256, 256, 256, 4]`, kernel size 3 throughout, and a receptive field
of **R = 13**. The four input/output channels carry the color-code detector data
(X/Z bases and color partitions) over the space–time syndrome volume.

Color codes select `code = "color"`; the validator restricts color to model ids
whose receptive field is ≤ 13 (ids **1, 2, 4, 5**). The Quick Start uses
model 5. The cell below builds the architecture from the registry and reports
its parameter count.


In [4]:
from types import SimpleNamespace

model_id = 5
spec = get_model_spec(model_id)

cfg_arch = SimpleNamespace(
    code="color", distance=13, n_rounds=13,
    model=SimpleNamespace(
        version="predecoder_memory_v1",
        num_filters=list(spec.num_filters), kernel_size=list(spec.kernel_size),
        dropout_p=0.0, activation="gelu", input_channels=4, out_channels=4,
    ),
)
m = ModelFactory.create_model(cfg_arch)
nparams = sum(p.numel() for p in m.parameters())

print(f"{'Model':<8} {'num_filters':<34} {'kernel_size':<24} {'RF':<6} {'num_params':>12}")
print("-" * 90)
print(f"  {model_id:<6} {str(spec.num_filters):<34} {str(spec.kernel_size):<24} "
      f"{spec.receptive_field:<6} {nparams:>12,}")


Model    num_filters                        kernel_size              RF       num_params
------------------------------------------------------------------------------------------
  5      [256, 256, 256, 256, 256, 4]       [3, 3, 3, 3, 3, 3]       13        7,134,468


---
## Training your own color-code pre-decoder

You might train your own color-code pre-decoder to target a different noise
profile, distance, or physical error rate. Color-code training reuses the same
launcher (`code/scripts/local_run.sh`) and the same Torch + cuStabilizer data
pipeline as the surface code, with a few color-specific differences:

- The global decoder is **Chromobius**, not PyMatching.
- The circuit is a **superdense, nearest-neighbor triangular color code** with
  Z feed-forward (`data.superdense`, `data.schedule`, `data.enable_z_feedforward`).
- **Noise upscaling** targets the color-code threshold (**4 × 10⁻³**) rather
  than the surface-code target (6 × 10⁻³).
- Training data is generated at a **fixed physical error rate** from a
  precomputed augmented DEM bundle (the surface code's `[p_min, p_max]` sweep is
  not yet available for color).

Color-code runs use **standalone configs** (`conf/config_color_*.yaml`) that
bypass the surface-only public-config validator; see the
[README](../README.md#color-code-support) for the authoritative list.


### Step 1 — precompute the augmented DEM bundle

The Torch color-code generator consumes a precomputed augmented DEM bundle
(produced by `qec.precompute_dem` with `--code color`). Generate it once per
`(distance, n_rounds, basis)` combination. The bundle is structural and is
reused across runs that differ only in the per-channel fault probability:

```bash
PYTHONPATH=code python code/qec/precompute_dem.py \
    --code color --distance 9 --n_rounds 9 --basis X \
    --dem_output_dir frames_data
PYTHONPATH=code python code/qec/precompute_dem.py \
    --code color --distance 9 --n_rounds 9 --basis Z \
    --dem_output_dir frames_data
```

> **This step is required for color codes.** Unlike the surface-code path
> (which falls back to generating frames in-memory when none are found), the
> color-code training driver does **not** auto-generate the bundle: it raises
> if `data.precomputed_frames_dir` is unset or the artifacts for the requested
> `(distance, n_rounds, basis)` are missing. Run the commands below before
> launching training in Step 2.


### Step 2 — launch training

Point the launcher at a color training config and set `WORKFLOW=train`. The
launcher forwards the config and `workflow.task` to `code/workflows/run.py`,
which dispatches to the color-code training entry point:

```bash
CONFIG_NAME=config_color_model_1_s_LR3e-4 \
    WORKFLOW=train \
    EXTRA_PARAMS="data.precomputed_frames_dir=$(pwd)/frames_data" \
    bash code/scripts/local_run.sh
```

**Shipped color-code configs:**

| Config file | Purpose |
|-------------|---------|
| `conf/config_color_model_1_s_LR3e-4.yaml` | Train a model-1-shaped color-code pre-decoder at `d = 9, r = 9` (superdense schedule). |
| `conf/config_color_threshold_model_1_d13.yaml` | Threshold sweep against a trained color-code checkpoint at `d = 13`. |
| `conf/config_inference_color_model_5.yaml` | Inference with a trained model-5-shaped color-code checkpoint (used in the Quick Start above; set `model_checkpoint_file`). |

The cell below prints the inference config that the Quick Start loaded.


In [5]:
config_path = os.path.join(PREDECODER_ROOT, "conf", "config_inference_color_model_5.yaml")
with open(config_path) as f:
    print(f.read())


# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# =============================================================================
# Color-code inference with a pre-trained Model-5-shaped checkpoint.
#
# Loads the checkpoint at `model_checkpoint_file` and runs the color-code
# inference path (`run_color` -> `count_logical_errors_color`).
#
# Usage:
#   PYTHONPATH=code 

---
## Evaluation: LER, SDR, and decode-time sweeps

The same public runner exposes the color-code evaluation workflows the model
card is built from. Drive them from the repository root by switching `WORKFLOW`:

```bash
# Logical-error-rate / threshold sweep over (distance, p)
CONFIG_NAME=config_inference_color_model_5 WORKFLOW=threshold \
    EXTRA_PARAMS="threshold.distances=[5,9,13,17,21] threshold.p_values=[0.001,0.002,0.003] threshold.num_samples=65536" \
    bash code/scripts/local_run.sh

# Syndrome-density-reduction sweep (input vs residual syndrome density)
CONFIG_NAME=config_inference_color_model_5 WORKFLOW=sdr bash code/scripts/local_run.sh

# Chromobius single-shot decode-time on original vs residual syndromes
CONFIG_NAME=config_inference_color_model_5 WORKFLOW=chromobius_timing bash code/scripts/local_run.sh
```

`WORKFLOW=inference` (the Quick Start path) is also available from the CLI for a
single `(distance, p, basis)` point:

```bash
CONFIG_NAME=config_inference_color_model_5 WORKFLOW=inference \
    EXTRA_PARAMS="test.num_samples=65536 test.p_error=0.001 test.meas_basis_test=both" \
    bash code/scripts/local_run.sh
```


---
## Optimizing inference (ONNX / TensorRT)

The color-code pre-decoder is the **same fully-convolutional architecture** as
the surface-code models, so it maps onto the identical deployment path: export
the network to ONNX, then compile a TensorRT engine. The surface-code tutorial
([`predecoder.ipynb`](./predecoder.ipynb)) walks through FP16 engine builds and
FP8 quantization in detail; those steps apply unchanged to the color CNN.

> **Roadmap:** the Color-Code Model 5 pre-decoder runs in fp32. INT8/FP8
> quantization and the fused full-pipeline ONNX export (preprocessing + CNN +
> residual assembly in one graph, as shipped for surface code) are being
> productized for color and will be documented here when available.

The cell below exports the raw color-code CNN to ONNX as a starting point. The
input is the space–time syndrome volume `(batch, channels=4, n_rounds, n_rows,
n_cols)`, where for a distance-`d` triangular color code `n_rows = d + (d-1)//2`
and `n_cols = d`.


In [ ]:
import logging, warnings
for _name in ["onnxscript", "onnx_ir", "torch.onnx", "torch"]:
    logging.getLogger(_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

# Color-code space-time volume matching the Quick Start window (d = n_rounds).
d, T = cfg.test.distance, cfg.test.n_rounds
n_rows, n_cols = d + (d - 1) // 2, d
example = torch.zeros(2, 4, T, n_rows, n_cols, dtype=torch.float32, device=device)
print(f"Input shape: {tuple(example.shape)}  (batch, channels, rounds, rows, cols)")

onnx_path = f"color_predecoder_d{d}_T{T}.onnx"
export_kwargs = dict(
    opset_version=18,
    input_names=["syndrome_volume"], output_names=["correction_logits"],
    dynamic_axes={"syndrome_volume": {0: "batch"}, "correction_logits": {0: "batch"}},
    do_constant_folding=True,
)
try:
    import onnx
    # PyTorch >= 2.9 defaults to the torch.export-based ONNX exporter, which needs
    # `onnxscript`. If that package isn't installed, fall back to the legacy
    # TorchScript exporter (ships with torch) -- sufficient for this plain CNN.
    try:
        torch.onnx.export(model, example, onnx_path, **export_kwargs)
        exporter = "torch.export (default)"
    except ImportError:
        torch.onnx.export(model, example, onnx_path, dynamo=False, **export_kwargs)
        exporter = "legacy TorchScript (pip install onnxscript for the default exporter)"
    onnx.checker.check_model(onnx.load(onnx_path))
    print(f"Exported and verified: {onnx_path} "
          f"({os.path.getsize(onnx_path) / 1024**2:.1f} MB)  [{exporter}]")
    print("Build a TensorRT engine from this graph following Step 2 of the "
          "surface-code tutorial (predecoder.ipynb).")
except ImportError:
    print("onnx not installed — skipping export. Install with: pip install onnx")
except Exception as e:
    print(f"ONNX export failed: {e}")

### Timing the exported ONNX graph

With the CNN exported, you can measure its forward-pass latency directly
from the ONNX graph using [ONNX Runtime](https://onnxruntime.ai/), before
investing in a TensorRT build. The cell below loads the graph, picks the
CUDA execution provider when `onnxruntime-gpu` is available (falling back to
CPU otherwise), warms up, and then reports mean / p50 / p99 latency over a
timed loop.

This measures the **pre-decoder CNN forward pass only** — it does not include
Chromobius decoding of the residual (use `WORKFLOW=chromobius_timing` above for
the global-decoder timing). For an apples-to-apples production number, build a
TensorRT engine from the same graph (Step 2 of the surface-code tutorial) and
time that; ONNX Runtime here is the quick, dependency-light first look.

In [ ]:
import time

BATCH = 256          # shots per inference call
WARMUP = 10          # untimed iterations to amortize allocation / autotuning
ITERS = 100          # timed iterations

try:
    import onnxruntime as ort

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    sess = ort.InferenceSession(onnx_path, providers=providers)
    active = sess.get_providers()[0]
    print(f"ONNX Runtime session on: {active}")

    # Random syndrome volume of the same shape the graph was exported with,
    # only the (dynamic) batch dimension differs.
    rng = np.random.default_rng(0)
    sample = rng.integers(0, 2, size=(BATCH, 4, T, n_rows, n_cols)).astype(np.float32)
    feed = {"syndrome_volume": sample}

    for _ in range(WARMUP):
        sess.run(None, feed)

    times_ms = []
    for _ in range(ITERS):
        t0 = time.perf_counter()
        sess.run(None, feed)
        times_ms.append((time.perf_counter() - t0) * 1e3)

    times_ms = np.array(times_ms)
    mean, p50, p99 = times_ms.mean(), np.percentile(times_ms, 50), np.percentile(times_ms, 99)
    per_shot_us = mean / BATCH * 1e3
    print(f"Batch {BATCH} | mean {mean:.3f} ms | p50 {p50:.3f} ms | p99 {p99:.3f} ms")
    print(f"  -> {per_shot_us:.2f} us/shot, {BATCH / mean * 1e3:,.0f} shots/s")
except ImportError:
    print("onnxruntime not installed — skipping timing. "
          "Install with: pip install onnxruntime-gpu (or onnxruntime for CPU).")
except NameError:
    print("onnx_path is undefined — run the ONNX export cell above first.")
except Exception as e:
    print(f"ONNX timing failed: {e}")

---
## Learn More

- **Color code support** in the repository [README](../README.md#color-code-support)
  — the authoritative reference for configs, the augmented-DEM precompute, and
  the inference/training/threshold/SDR/timing workflows.
- **Cluster / remote training** — [TRAINING.md](../TRAINING.md) (color-code
  training section).
- **Surface-code tutorial** — [`predecoder.ipynb`](./predecoder.ipynb) for the
  pre-decoder concept, the four-strategy comparison, and the full ONNX → TensorRT
  FP16 → FP8 optimization walkthrough.
- **Chromobius** — the color-code global decoder:
  <https://github.com/quantumlib/chromobius>.
- **Paper** — Chamberland, Olle, Li, Thornton, Baratta, *Fast and accurate
  AI-based pre-decoders for surface codes*,
  [arXiv:2604.12841](https://arxiv.org/abs/2604.12841).
